# **Load Libraries**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# **Path Configuration and Output Directory Creation**

In [ ]:
hazy_input_dir = "Datset/01. Hazy - Raw"
lowlight_input_dir = "Datset/02. Low Light - Raw"

hazy_output_dir = "output/hazy- student enhanced"
lowlight_output_dir = "output/lowlight- student enhanced"

os.makedirs(hazy_output_dir, exist_ok=True)
os.makedirs(lowlight_output_dir, exist_ok=True)

# **Hazy Image Enhancement Pipeline**

In [ ]:
def enhance_hazy(img, gamma=2.2, sat_gain=1.15, sharp_amount=0.12):

    # ------- 1. Gamma correction (per channel) -------
    img_f = img.astype(np.float32) / 255.0
    img_gamma = np.power(np.clip(img_f, 0, 1), gamma)
    img_gamma = np.uint8(np.clip(img_gamma * 255.0, 0, 255))

    # ------- 2. Lightness-based contrast enhancement in LAB -------
    lab = cv2.cvtColor(img_gamma, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)

    # Stretch L to use full [0, 255] 
    Lf = L.astype(np.float32)
    L_stretch = (Lf - Lf.min()) / (Lf.max() - Lf.min() + 1e-6) * 255.0
    L_stretch = np.uint8(np.clip(L_stretch, 0, 255))

    # Tiny unsharp mask on L 
    blur = cv2.GaussianBlur(L_stretch, (0, 0), 1.0)
    L_sharp = cv2.addWeighted(L_stretch,
                              1.0 + sharp_amount,
                              blur,
                              -sharp_amount,
                              0)

    # Reconstruct LAB image and convert back to BGR
    lab_enh = cv2.merge([L_sharp, A, B])
    bgr_contrast = cv2.cvtColor(lab_enh, cv2.COLOR_LAB2BGR)

    # ------- 3. Small saturation boost -------
    hsv = cv2.cvtColor(bgr_contrast, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 1] *= sat_gain     
    hsv[..., 1] = np.clip(hsv[..., 1], 0, 255)
    vivid = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    return vivid

# **Low-Light Image Enhancement Pipeline**

In [ ]:
def enhance_low_light(img, factor=3.0, gamma=0.7):
    #  ------- 1. Convert to HSV  ------- 
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)

    #  ------- 2. Brightness enhancement on V channel ------- 
    Vf = V.astype(np.float32) / 255.0
    Vf = np.power(Vf, gamma)
    Vf = np.clip(Vf * factor, 0, 1)
    V_new = (Vf * 255).astype(np.uint8)

    #  ------- 3. Reduce saturation slightly -------  
    S = (S * 0.95).astype(np.uint8)

    #  ------- 4. Reconstruct HSV and convert back to BGR ------- 
    hsv_new = cv2.merge([H, S, V_new])
    bgr = cv2.cvtColor(hsv_new, cv2.COLOR_HSV2BGR)

    #  ------- 5. Denoise ------- 
    denoised = cv2.fastNlMeansDenoisingColored(bgr, None, h=5, hColor=5, templateWindowSize=7, searchWindowSize=21)

    return denoised

# **Image Enhancement Process**

In [ ]:
# Loop through all image files in the hazy input directory
for filename in os.listdir(hazy_input_dir):
    if filename.lower().endswith(('.png')):
        img_path = os.path.join(hazy_input_dir, filename)
        
        # Read input image
        img = cv2.imread(img_path)

        # Skip file if image cannot be read
        if img is None:
            print(f"❌ Cannot read {filename}")
            continue

        # Apply hazy image enhancement pipeline
        enhanced = enhance_hazy(img)

        # Save enhanced image to output directory
        save_path = os.path.join(hazy_output_dir, filename)
        cv2.imwrite(save_path, enhanced)

print("✅ Hazy images enhancement done.")

In [ ]:
# Loop through all image files in the low-light input directory
for filename in os.listdir(lowlight_input_dir):
    if filename.lower().endswith(('.png')):
        img_path = os.path.join(lowlight_input_dir, filename)

        # Read input image
        img = cv2.imread(img_path)

        # Skip file if image cannot be read
        if img is None:
            print(f"❌ Cannot read {filename}")
            continue

        # Apply low-light image enhancement pipeline
        enhanced = enhance_low_light(img)

        # Save enhanced image to output directory
        save_path = os.path.join(lowlight_output_dir, filename)
        cv2.imwrite(save_path, enhanced)

print("✅ Low-light images enhancement done.")

# **Before/After Image Comparison**

In [ ]:
def list_images(folder):
    """
    Returns a sorted list of image filenames in the given folder.
    Only files with .png extension are included.
    """
    exts = (".png")
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(exts)])

def read_rgb(path):
    """
    Reads an image from disk and converts it from BGR to RGB format.
    """
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_triplet(raw, enhanced, gt, title=None):
    """
    Displays a triplet of images: raw input, enhanced result,
    and ground truth for qualitative comparison.
    """
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    imgs = [raw, enhanced, gt]
    titles = ["RAW", "MY ENHANCED", "GROUND TRUTH"]

    for ax, img, t in zip(axs, imgs, titles):
        if img is not None:
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "Missing",
                    ha="center", va="center", fontsize=14)
        ax.set_title(t)
        ax.axis("off")

    if title:
        fig.suptitle(title, fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visual comparison of raw, enhanced, and ground-truth hazy images

hazy_raw_dir = "Datset/01. Hazy - Raw"
hazy_enh_dir = "output/hazy- student enhanced"
hazy_gt_dir  = "Datset/01..Hazy - Enhanced (GT)"

hazy_files = list_images(hazy_raw_dir)

print(f"Displaying {len(hazy_files)} hazy images...\n")

for fname in hazy_files:
    raw = read_rgb(os.path.join(hazy_raw_dir, fname))
    enh = read_rgb(os.path.join(hazy_enh_dir, fname))
    gt  = read_rgb(os.path.join(hazy_gt_dir, fname))

    show_triplet(raw, enh, gt, title=f"Hazy Image: {fname}")

In [ ]:
# Visual comparison of raw, enhanced, and ground-truth low-light images

low_raw_dir = "Datset/02. Low Light - Raw"
low_enh_dir = "output/lowlight- student enhanced"
low_gt_dir  = "Datset/02.. Low Light - Enhanced (GT)"

low_files = list_images(low_raw_dir)

print(f"Displaying {len(low_files)} low-light images...\n")

for fname in low_files:
    raw = read_rgb(os.path.join(low_raw_dir, fname))
    enh = read_rgb(os.path.join(low_enh_dir, fname))
    gt  = read_rgb(os.path.join(low_gt_dir, fname))

    show_triplet(raw, enh, gt, title=f"Low-Light Image: {fname}")

# **Before/After Histograms**

In [ ]:
def plot_histogram(raw_path, enhanced_path, title=""):
    """
    Plot RAW and Enhanced histograms using HSV V-channel.
    """

    raw = cv2.imread(raw_path)
    enhanced = cv2.imread(enhanced_path)

    if raw is None or enhanced is None:
        print("Error: Image not found.")
        return

    # Extract V channel
    raw_v = cv2.cvtColor(raw, cv2.COLOR_BGR2HSV)[..., 2].ravel()
    enh_v = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)[..., 2].ravel()

    plt.figure(figsize=(7,4))

    plt.hist(raw_v, bins=256, range=(0,256), alpha=0.5, label="RAW")
    plt.hist(enh_v, bins=256, range=(0,256), alpha=0.5, label="Enhanced")

    # mean intensity lines
    plt.axvline(np.mean(raw_v), linestyle="--", label=f"RAW mean = {np.mean(raw_v):.1f}")
    plt.axvline(np.mean(enh_v), linestyle="--", label=f"Enhanced mean = {np.mean(enh_v):.1f}")

    plt.xlabel("Intensity Value (0–255)")
    plt.ylabel("Pixel Count")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot RAW vs Enhanced histograms for all hazy images

raw_dir = "Datset/01. Hazy - Raw"
enh_dir = "output/hazy- student enhanced"

valid_ext = (".png")

files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(valid_ext)])

print(f"Found {len(files)} images")

for fname in files:
    raw_path = os.path.join(raw_dir, fname)
    enh_path = os.path.join(enh_dir, fname)

    if not os.path.exists(enh_path):
        print(f"⚠️ Enhanced missing for {fname}, skipped")
        continue

    plot_histogram(
        raw_path,
        enh_path,
        title=f"Hazy Histogram: {fname}"
    )

In [ ]:
# Plot RAW vs Enhanced histograms for all low-light images

raw_dir = "Datset/02. Low Light - Raw"
enh_dir = "output/lowlight- student enhanced"

valid_ext = (".png")

files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(valid_ext)])

print(f"Found {len(files)} images")

for fname in files:
    raw_path = os.path.join(raw_dir, fname)
    enh_path = os.path.join(enh_dir, fname)

    if not os.path.exists(enh_path):
        print(f"⚠️ Enhanced missing for {fname}, skipped")
        continue

    plot_histogram(
        raw_path,
        enh_path,
        title=f"Low-Light Histogram: {fname}"
    )

# **Full-Reference Metrics (PSNR & SSIM)**

In [ ]:
VALID_EXT = (".png")

def evaluate_folder(enhanced_dir, gt_dir, category_name):
    enhanced_files = sorted([f for f in os.listdir(enhanced_dir) if f.lower().endswith(VALID_EXT)])

    psnr_list = []
    ssim_list = []

    print(f"\n==============================")
    print(f" Category: {category_name}")
    print(f" Enhanced: {enhanced_dir}")
    print(f" GT:       {gt_dir}")
    print(f" Found {len(enhanced_files)} enhanced images")
    print(f"==============================\n")

    for fname in enhanced_files:
        enhanced_path = os.path.join(enhanced_dir, fname)
        gt_path = os.path.join(gt_dir, fname)

        if not os.path.exists(gt_path):
            print(f"⚠️ {fname} | GT missing -> skipped")
            continue

        enhanced_img = cv2.imread(enhanced_path)
        gt_img = cv2.imread(gt_path)

        if enhanced_img is None or gt_img is None:
            print(f"❌ {fname} | cannot read -> skipped")
            continue

        if enhanced_img.shape != gt_img.shape:
            gt_img = cv2.resize(gt_img, (enhanced_img.shape[1], enhanced_img.shape[0]), interpolation=cv2.INTER_AREA)

        # Grayscale for PSNR/SSIM (standard)
        enhanced_gray = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2GRAY)
        gt_gray = cv2.cvtColor(gt_img, cv2.COLOR_BGR2GRAY)

        # Calculate PSNR & SSIM
        psnr_val = peak_signal_noise_ratio(gt_gray, enhanced_gray)
        ssim_val = structural_similarity(gt_gray, enhanced_gray)

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)

        print(f"{fname:<25} | PSNR: {psnr_val:>6.2f} dB | SSIM: {ssim_val:>7.4f}")

    # Averages
    print("\n------ SUMMARY ------")
    if len(psnr_list) > 0:
        print(f"Average PSNR ({category_name}): {np.mean(psnr_list):.2f} dB")
        print(f"Average SSIM ({category_name}): {np.mean(ssim_list):.4f}")
        print(f"Valid pairs evaluated: {len(psnr_list)}")
    else:
        print("❌ No valid image pairs evaluated.")
    print("---------------------\n")

    return psnr_list, ssim_list


# =========================
# PATHS 
# =========================
hazy_enhanced_dir = "output/hazy- student enhanced"
hazy_gt_dir      = "Datset/01..Hazy - Enhanced (GT)"

low_enhanced_dir  = "output/lowlight- student enhanced"
low_gt_dir       = "Datset/02.. Low Light - Enhanced (GT)"


# =========================
# RUN EVALUATION
# =========================
hazy_psnr, hazy_ssim = evaluate_folder(hazy_enhanced_dir, hazy_gt_dir, "Hazy")
low_psnr,  low_ssim  = evaluate_folder(low_enhanced_dir,  low_gt_dir,  "Low-Light")

# **No-Reference Quality Score (Sharpness & Contrast)**

In [ ]:
import os
import cv2
import numpy as np

VALID_EXT = (".png")

def sharpness_var_laplacian(bgr_img):
    """Sharpness = Variance of Laplacian on grayscale."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(lap.var())

def contrast_std_intensity(bgr_img):
    """Contrast = Std Dev of intensity on grayscale."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    return float(gray.std())

def evaluate_no_reference(enhanced_dir, gt_dir, category_name):
    """
    No-reference quality evaluation.
    Sharpness and contrast are computed independently for:
    - Ground-truth enhanced images
    - Student-enhanced images
    """
    files = sorted([f for f in os.listdir(enhanced_dir) if f.lower().endswith(VALID_EXT)])

    rows = []
    sharp_gt_list, sharp_enh_list = [], []
    cont_gt_list, cont_enh_list = [], []

    print(f"\n==============================")
    print(f" Category: {category_name}")
    print(f" Enhanced: {enhanced_dir}")
    print(f" GT:       {gt_dir}")
    print(f" Found {len(files)} student images")
    print(f"==============================\n")

    for fname in files:
        enh_path = os.path.join(enhanced_dir, fname)
        gt_path  = os.path.join(gt_dir, fname)

        if not os.path.exists(gt_path):
            print(f"⚠️ {fname} | GT missing -> skipped")
            continue

        enh_img = cv2.imread(enh_path)
        gt_img  = cv2.imread(gt_path)

        if enh_img is None or gt_img is None:
            print(f"❌ {fname} | cannot read -> skipped")
            continue

        if enh_img.shape != gt_img.shape:
            gt_img = cv2.resize(gt_img, (enh_img.shape[1], enh_img.shape[0]), interpolation=cv2.INTER_AREA)

        # Compute metrics
        sharp_gt  = sharpness_var_laplacian(gt_img)
        sharp_enh = sharpness_var_laplacian(enh_img)

        cont_gt   = contrast_std_intensity(gt_img)
        cont_enh  = contrast_std_intensity(enh_img)

        rows.append((fname, sharp_gt, sharp_enh, cont_gt, cont_enh))

        sharp_gt_list.append(sharp_gt)
        sharp_enh_list.append(sharp_enh)
        cont_gt_list.append(cont_gt)
        cont_enh_list.append(cont_enh)

        print(f"{fname:<22} | Sharp(GT): {sharp_gt:>10.2f} | Sharp(Enhanced): {sharp_enh:>10.2f} "
              f"| Contrast(GT): {cont_gt:>7.2f} | Contrast(Enhanced): {cont_enh:>7.2f}")

    print("\n------ AVERAGES ------")
    if rows:
        print(f"Avg Sharpness (GT)       [{category_name}]: {np.mean(sharp_gt_list):.2f}")
        print(f"Avg Sharpness (Enhanced) [{category_name}]: {np.mean(sharp_enh_list):.2f}")
        print(f"Avg Contrast  (GT)       [{category_name}]: {np.mean(cont_gt_list):.2f}")
        print(f"Avg Contrast  (Enhanced) [{category_name}]: {np.mean(cont_enh_list):.2f}")
        print(f"Valid pairs evaluated: {len(rows)}")
    else:
        print("❌ No valid image pairs evaluated.")
    print("----------------------\n")

    return rows

In [ ]:
# HAZY
hazy_enhanced_dir = "output/hazy- student enhanced"
hazy_gt_dir      = "Datset/01..Hazy - Enhanced (GT)"
hazy_rows = evaluate_no_reference(hazy_enhanced_dir, hazy_gt_dir, "Hazy")

# LOW-LIGHT
low_enhanced_dir  = "output/lowlight- student enhanced"
low_gt_dir       = "Datset/02.. Low Light - Enhanced (GT)"
low_rows = evaluate_no_reference(low_enhanced_dir, low_gt_dir, "Low-Light")